# ♻️ Laporan Akhir: Replikasi Penelitian Deteksi Sampah Daur Ulang YOLOv8

**Nama Pengkaji: Fadli Hifizansyah**  
**NIM: 241730042**  
**Kelas: Informatika 4B**  
**Mata Kuliah: Kecerdasan Buatan**  

---

## 📌 1. Tujuan dan Ringkasan Proyek
Proyek ini bertujuan untuk melakukan **replikasi penelitian object detection** berbasis Deep Learning **YOLOv8** untuk mendeteksi sampah daur ulang. 

### Referensi Jurnal Acuan:
> **Visen, & Charibaldi, N. (2025). Penerapan Object Detection Menggunakan Deep Learning YOLOv8 Untuk Mengidentifikasi Sampah Anorganik (Maksimal Sepuluh Objek) Dalam Satu Citra. Jurnal Teknologi Informasi dan Ilmu Komputer (JTIIK), 12(1), 195–202.**

### Aspek Utama Replikasi:
- **Dataset Baru:** Menggunakan dataset **Recyclable Waste** dari Roboflow (berukuran besar, total 19.493 gambar) alih-alih dataset asli jurnal, guna melihat efektivitas model pada data bervariasi.
- **Kategori Deteksi (4 Kelas):** `Kaca`, `Kertas`, `Logam`, dan `Plastik`.
- **Dua Varian Model:** Menguji dan membandingkan **YOLOv8n (Nano)** dan **YOLOv8s (Small)**.
- **Hardware Lokal:** Menggunakan **NVIDIA GeForce RTX 4050 Laptop GPU** (VRAM 6GB).

## 📂 2. Konteks Repositori dan Struktur Folder
Struktur folder dirancang secara modular agar mudah dibaca dan dikembangkan lebih lanjut:
```text
deteksi-sampah-replikasi/
├── 05_Source_Code/
│   ├── Notebook/              # File eksperimen (.ipynb)
│   │   ├── preprocessing.ipynb
│   │   ├── training.ipynb
│   │   └── evaluation.ipynb
│   └── Script/                # Script otomasi (.py)
│       ├── preprocessing.py
│       ├── train.py
│       ├── test.py
│       └── predict.py
├── citra_uji_eksternal/       # Foto uji di luar dataset
├── train/, valid/, test/      # Dataset gambar & label
├── runs/detect/               # Log & bobot model (best.pt)
├── data.yaml                  # Konfigurasi kelas & folder YOLO
└── analysis.ipynb             # Notebook laporan utama (Sel ini!)
```

## 🛠️ 3. Setup Environment dan Import Library
Jalankan sel ini untuk memastikan semua dependensi utama siap digunakan.

In [ ]:
import os
import yaml
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from ultralytics import YOLO
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 📝 4. Konfigurasi Utama & Validasi Dataset
Memvalidasi letak dataset dan memastikan path absolut tersimpan di `data.yaml` agar tidak terjadi error pembacaan folder.

In [ ]:
project_root = os.getcwd()
yaml_path = os.path.join(project_root, "data.yaml")

if os.path.exists(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data_yaml = yaml.safe_load(f)
    
    # Set path absolut
    data_yaml['path'] = project_root.replace('\\', '/')
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
        
    print("✅ data.yaml terverifikasi dan diperbarui secara dinamis!")
    print("Daftar Kelas:", data_yaml.get('names'))
else:
    print("❌ File data.yaml tidak ditemukan di root proyek!")

## 📊 5. Analisis Dataset (EDA Ringkas)
Mari kita hitung distribusi jumlah file gambar di setiap split (`train`, `valid`, `test`) guna memastikan kebenaran pemisahan dataset.

In [ ]:
splits = ['train', 'valid', 'test']
counts = {}
for split in splits:
    img_dir = os.path.join(project_root, split, 'images')
    if os.path.exists(img_dir):
        img_files = glob.glob(os.path.join(img_dir, "*"))
        # Filter gambar
        img_files = [f for f in img_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
        counts[split] = len(img_files)
    else:
        counts[split] = 0

print("📈 Statistik Jumlah Citra per Split Dataset:")
for k, v in counts.items():
    print(f"   - {k.capitalize()}: {v} gambar")

# Visualisasi sederhana menggunakan Matplotlib
plt.figure(figsize=(7, 4))
colors = ['#4f46e5', '#0ea5e9', '#10b981']
plt.bar(counts.keys(), counts.values(), color=colors, width=0.5)
plt.title('Proporsi Citra dalam Dataset', fontweight='bold', fontsize=12)
plt.ylabel('Jumlah Gambar')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

## 🏆 6. Perbandingan Performa Model (YOLOv8n vs YOLOv8s)
Dalam replikasi ini, kedua model telah dilatih selama **60 epoch** dengan ukuran gambar **640x640**.
Berikut adalah tabel perbandingan hasil evaluasi metrik performa formal pada **Dataset Test** (737 gambar):

| Model Varian | Model Size | Precision | Recall | mAP50 | mAP50-95 | Keterangan / Keunggulan |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **YOLOv8n (Nano)** | ~6.2 MB | **0.957 (95.7%)** | **0.887 (88.7%)** | **0.915 (91.5%)** | **0.830 (83.0%)** | Inferensi super cepat, sangat ringan, cocok untuk perangkat mobile/TFLite. |
| **YOLOv8s (Small)** | ~22.5 MB | **0.959 (95.9%)** | **0.892 (89.2%)** | **0.923 (92.3%)** | **0.842 (84.2%)** | Akurasi sedikit lebih tinggi (+0.8% mAP50), komputasi lebih besar. |

> **💡 Analisis Metrik:** 
> YOLOv8s memiliki akurasi (mAP50) sebesar **92.3%**, sedikit mengungguli YOLOv8n (**91.5%**). Namun, ukuran file YOLOv8s hampir **4 kali lipat lebih besar** dibandingkan YOLOv8n. Untuk penggunaan praktis sehari-hari, **YOLOv8n** dinilai jauh lebih efisien karena kecepatan inferensi yang sangat tinggi dengan pengorbanan akurasi yang minimal.

## 📊 7. Visualisasi Kurva Evaluasi
Mari tampilkan **Confusion Matrix** dari model YOLOv8n terlatih untuk menganalisis kelas mana yang paling sering mengalami kesalahan deteksi (*misclassification*).

In [ ]:
cm_path = os.path.join(project_root, "runs", "detect", "train-yolov8n", "confusion_matrix.png")

if os.path.exists(cm_path):
    print("📊 CONFUSION MATRIX (YOLOv8n):")
    display(Image(filename=cm_path, width=600))
else:
    # Cari di training folder lain jika path berbeda
    fallback_cm = glob.glob(os.path.join(project_root, "runs", "detect", "*", "confusion_matrix.png"))
    if fallback_cm:
        print(f"📊 CONFUSION MATRIX ({os.path.basename(os.path.dirname(fallback_cm[0]))}):")
        display(Image(filename=fallback_cm[0], width=600))
    else:
        print("⚠️ confusion_matrix.png tidak ditemukan. Silakan jalankan validasi terlebih dahulu.")

## 🔮 8. Uji Deteksi Citra Nyata (Inferensi Eksternal)
Di bawah ini kita jalankan inferensi menggunakan model YOLOv8n terbaik pada gambar di folder `citra_uji_eksternal` dan menampilkan hasilnya.

In [ ]:
weights_n = os.path.join(project_root, "runs", "detect", "train-yolov8n", "weights", "best.pt")
test_images_dir = os.path.join(project_root, "citra_uji_eksternal")

if os.path.exists(weights_n) and os.path.exists(test_images_dir):
    model_n = YOLO(weights_n)
    print("🚀 Menjalankan deteksi pada citra uji eksternal...")
    results = model_n.predict(source=test_images_dir, save=True, conf=0.25, verbose=False)
    
    # Cari folder predict terbaru
    predict_folders = glob.glob(os.path.join(project_root, "runs", "detect", "predict*"))
    if predict_folders:
        latest_predict = max(predict_folders, key=os.path.getmtime)
        print(f"Citra hasil prediksi disimpan di: {latest_predict}\n")
        
        # Tampilkan gambar hasil prediksi secara visual
        predicted_images = glob.glob(os.path.join(latest_predict, "*"))
        for p_img in predicted_images:
            if p_img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                display(Image(filename=p_img, width=450))
                print("-" * 50)
else:
    print("⚠️ Bobot YOLOv8n atau folder citra uji tidak ditemukan.")

## 📝 9. Kesimpulan, Keterbatasan, & Langkah Selanjutnya

### Kesimpulan:
1. **Replikasi Sukses:** Replikasi jurnal Visen & Charibaldi (2025) berhasil diterapkan pada dataset baru (Recyclable Waste) dengan hasil akurasi yang memuaskan.
2. **Akurasi Tinggi:** Model YOLOv8n dan YOLOv8s mencatat nilai mAP50 di atas **91%**, menunjukkan ketangguhan arsitektur YOLOv8 dalam mengidentifikasi jenis-jenis sampah daur ulang.
3. **Kemampuan Multi-Object:** Model mampu mendeteksi hingga **16 objek sampah dalam satu citra** secara bersamaan, melebihi batasan deteksi maksimal 10 objek pada penelitian asli.

### Keterbatasan Penelitian:
- **Variasi Sampah Organik:** Dataset ini didominasi oleh sampah anorganik daur ulang (kaca, kertas, logam, plastik) dan belum mencakup sampah organik (sisa makanan, daun, dll).
- **Kondisi Cahaya & Blur:** Akurasi dapat menurun jika kualitas gambar buruk, blur, atau dalam pencahayaan yang sangat redup.

### Rekomendasi / Langkah Selanjutnya:
1. **Ekspansi Kelas:** Menambahkan kelas baru untuk sampah organik atau sampah medis (B3).
2. **Implementasi Edge Device:** Mengekspor model terbaik ke format TensorFlow Lite (`.tflite`) atau ONNX untuk dideploy ke aplikasi Android/Raspberry Pi guna deteksi real-time.
3. **Integrasi Kamera Real-time:** Menghubungkan model dengan live webcam/CCTV untuk sistem pemilahan sampah otomatis di tempat umum.